# NCTBench-Pilot — Step 5: QA Generation
Stratified logarithmic sampling across subject x language x class.
Llama-3.1-8B-Instruct-Turbo via Together AI.
2 QA pairs per chunk. Evidence field required.

In [ ]:
import sys, json
from pathlib import Path
from collections import Counter
sys.path.insert(0, str(Path(r"E:\CSAA Project\pipeline")))
from config import CHUNKS_DIR, QA_DIR, TOGETHER_MODEL_PRIMARY

## Stratified Sampling Allocation

In [ ]:
chunks = json.loads(
    (CHUNKS_DIR / "all_chunks.json").read_text(encoding="utf-8")
)
from step5_qa_generate import build_stratified_sample
sample = build_stratified_sample(chunks, total=99999)
print(f"Total sample: {len(sample)} chunks")
print("By language:", dict(Counter(c["language"] for c in sample)))
print("By subject :", dict(Counter(c["subject_code"] for c in sample)))
print("By class   :", dict(sorted(
    Counter(c["class_num"] for c in sample).items()
)))

## Prompt Templates

In [ ]:
from step5_qa_generate import build_prompt
bn = next(c for c in chunks if c["language"] == "bn")
en = next(c for c in chunks if c["language"] == "en")
print("=== BENGALI PROMPT (first 600 chars) ===")
print(build_prompt(bn, 2)[:600])
print("\n=== ENGLISH PROMPT (first 600 chars) ===")
print(build_prompt(en, 2)[:600])

## Run QA Generation

In [ ]:
from step5_qa_generate import run_qa_generation
run_qa_generation(day=1)

In [ ]:
qa_files = sorted(QA_DIR.glob("qa_*.json"))
all_pairs = []
for f in qa_files:
    pairs = json.loads(f.read_text(encoding="utf-8"))
    if pairs:
        all_pairs.extend(pairs)
print(f"Chunk files processed : {len(qa_files)}")
print(f"Total QA pairs        : {len(all_pairs)}")
bn_p = [p for p in all_pairs if p["language"] == "bn"]
en_p = [p for p in all_pairs if p["language"] == "en"]
print(f"Bengali pairs         : {len(bn_p)}")
print(f"English pairs         : {len(en_p)}")

In [ ]:
print("=== SAMPLE BENGALI QA PAIR ===")
if bn_p:
    p = bn_p[0]
    print(f"Q: {p['question']}")
    print(f"A: {p['answer']}")
    print(f"E: {p.get('evidence','')[:200]}")
print("\n=== SAMPLE ENGLISH QA PAIR ===")
if en_p:
    p = en_p[0]
    print(f"Q: {p['question']}")
    print(f"A: {p['answer']}")
    print(f"E: {p.get('evidence','')[:200]}")